# SHQ Analysis Program 3 - Double Leg Drop Landing Knee Flexion
Updated 04/27/2026

-------
## Cell 1 - Imports
Import the necessary packages to run the program.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from tkinter import Tk
from tkinter.filedialog import askdirectory, asksaveasfilename, askopenfilename
from IPython.display import display

print(f'{datetime.now()} - Imports OK')

# also in this cell, we are defining custom theme for plots. 
custom_theme = {
    "axes.spines.right":  False,
    "axes.spines.top":    False,
    "axes.titlelocation": "left",
    "axes.titley":        1,
    "font.weight":        "bold",
    "axes.titlesize":     "x-large",
    "axes.labelsize":     "x-large",
    "axes.titleweight":   "bold",
    "axes.labelweight":   "bold"
}

plt.rcParams.update({
    **custom_theme,
    'figure.facecolor': 'white',
    'axes.labelcolor':  'black',
    'axes.edgecolor':   'black',
    'xtick.color':      'black',
    'ytick.color':      'black',
    'axes.titlecolor':  'black'
})

2026-04-28 10:39:52.045296 - Imports OK


--------
## Cell 2 - Define Functions

`pick_directory` -- opens a folder browser dialog to find the directory in which the files are stsored

`pick_save_path` -- opens a Save As dialog for saving the results to 

`pick_file` -- allows you to choose the file with the frames information in it

In [2]:
# Defining pick_directory function
def pick_directory(prompt = "Select Participant's Data Folder"):
    root = Tk()
    root.attributes('-topmost', True)
    root.after(1, root.attributes, '-topmost', False)
    folder = askdirectory(title = prompt)
    root.destroy()
    return folder

# Defining pick_save_path folder
def pick_save_path(default_name):
    root = Tk()
    root.attributes('-topmost', True)
    root.after(1, root.attributes, '-topmost', False)
    path = asksaveasfilename(
        title = 'Choose where to save the drop landing results',
        initialfile = default_name,
        defaultextension = ".xlsx",
        filetypes = [('Excel files', "*.xlsx")]
    )
    root.destroy()
    return path

# Defining pick file function
def pick_file(prompt = 'Select the frames file', filetypes = [('CSV files', "*.csv")]):
    root = Tk()
    root.attributes('-topmost', True)
    root.after(1, root.attributes, '-topmost', False)
    path = askopenfilename(title = prompt, filetypes = filetypes)
    root.destroy()
    return path

print(f'{datetime.now()} - Functions defined OK')

2026-04-28 10:40:14.485092 - Functions defined OK


----------
## Cell 3 - Select the master frames file and root data folder

Two dialogs will open:
1. Select the **master frames .csv file** ('SHQ_DL_Drop_Frames.csv'). 
2. Select the **root data collection folder** - this will find the participant's drop landing files. 

In [3]:
# Choose the file with the frames information in it
frames_file = pick_file(prompt = 'Select SHQ_DL_Drop_Frames.csv')
frames_dat = pd.read_csv(frames_file)

print(f'Frames file located: {os.path.basename(frames_file)}')
print(f"{len(frames_dat)} trials across {frames_dat['ID'].nunique()} participants.")

# Now select the data collection folder
data_root = pick_directory(prompt = 'Select the Data Collection folder with participant subfolders')
print(f'\nData root folder: {data_root}')

Frames file located: SHQ_DL_Drop_Frames.csv
64 trials across 22 participants.

Data root folder: E:/Research/0-Mentoring/MoosbruggerJ/Data Collection


--------
## Cell 4 - Primary Analysis Cell to Extract Knee Flexion Angles
For each trial in the `frames_dat` dataframe, this code will:
1. Navigate to the participant's subolfder and find the matching csv files for the drop landing.

In [9]:
results = []
missing = []

for _, row in frames_dat.iterrows():
    participant_id = row['ID']
    trial_name = row['Trial']
    contact_frame = row['Second_Leg_Contact_Frame']
    
    trial_file = os.path.join(data_root, participant_id, 'Full Data', f'{trial_name}.csv')

    if not os.path.exists(trial_file):
        print(f'File not found for {trial_file}, skipping trial.')
        missing.append({'ID': participant_id,
                        'Trial': trial_name,
                        'Reason': 'File not found'})
        continue

    # Load in the file
    try:
        participant_dat = pd.read_csv(trial_file)
    except Exception as e:
        print(f'Error reading {trial_file}: {e}')
        missing.append({'ID': participant_id, 
                        'Trial': trial_name,
                        'Reason': f'Read error: {e}'})
        continue
    
    # Sanity check, make sure the contact frame index is within the bounds (# rows) of the file
    if contact_frame > len(participant_dat):
        print(f' Contact frame is incorrect for {os.path.basename(trial_file)}')
        missing.append({'ID': participant_id,
                        'Trial': trial_name,
                        'Reason': "incorrect contact frame"})

    # Primary results filling
    # Pull left & right knee angles from `participant_dat`
    time_s = participant_dat['time_s']
    left_knee_angle = participant_dat['left_knee_angle_deg']
    right_knee_angle = participant_dat['right_knee_angle_deg']
    
    # Pull contact times
    left_knee_contact = float(left_knee_angle[contact_frame])
    right_knee_contact = float(right_knee_angle[contact_frame])

    # Subset knee angles 
    left_knee_angle_post = left_knee_angle[contact_frame:]
    right_knee_angle_post = right_knee_angle[contact_frame:]

    # Pull Peak Angles
    left_knee_peak = float(np.nanmax(left_knee_angle_post))
    right_knee_peak = float(np.nanmax(right_knee_angle_post))

    # Knee Excursion
    left_knee_excursion = float(left_knee_peak - left_knee_contact)
    right_knee_excursion = float(right_knee_peak - right_knee_contact)

    # For sanity checking, could also plot
    #print(f'{participant_id}_{trial_name}, LEFT = {left_knee_excursion}, RIGHT = {right_knee_excursion}')

    # Add the participant's trial to the Results list
    results.append({
        'ID': participant_id,
        'Trial': trial_name,
        'Left_Contact_Deg': round(left_knee_contact, 4),
        'Left_Peak_Deg': round(left_knee_peak, 4),
        'Left_Excursion_Deg': round(left_knee_excursion, 4),
        'Right_Contact_Deg': round(right_knee_contact, 4),
        'Right_Peak_Deg': round(right_knee_peak, 4),
        'Right_Excursion_Deg': round(right_knee_excursion, 4)
    })

results_dat = pd.DataFrame(results)

print(f'\nDone: {len(results)} trials processed, {len(missing)} not processed.')

display(results_dat)


File not found for E:/Research/0-Mentoring/MoosbruggerJ/Data Collection\SHQ018\Full Data\Double_drop_2.csv, skipping trial.
File not found for E:/Research/0-Mentoring/MoosbruggerJ/Data Collection\SHQ018\Full Data\Double_drop_3.csv, skipping trial.

Done: 62 trials processed, 2 not processed.


,ID,Trial,Left_Contact_Deg,Left_Peak_Deg,Left_Excursion_Deg,Right_Contact_Deg,Right_Peak_Deg,Right_Excursion_Deg
0,SHQ001,Double_drop_1,40.2251,82.2780,42.0529,44.0786,77.9267,33.8481
1,SHQ001,Double_drop_2,19.7800,80.8605,61.0805,25.3088,83.4828,58.1740
2,SHQ001,Double_drop_3,23.0460,88.5583,65.5123,26.3041,91.5979,65.2939
3,SHQ002,Double_drop_1,46.0000,66.7963,20.7963,28.3870,66.4652,38.0782
4,SHQ002,Double_drop_2,53.1826,68.3347,15.1521,33.5196,69.5056,35.9860
...,...,...,...,...,...,...,...,...
57,SHQ021,Double_drop_2,35.0297,92.5102,57.4804,35.8187,92.8301,57.0114
58,SHQ021,Double_drop_3,41.0062,98.4308,57.4246,40.6825,101.0743,60.3918
59,SHQ022,Double_drop_1,29.8513,92.5240,62.6727,33.2738,93.5580,60.2842
60,SHQ022,Double_drop_2,31.3626,98.3164,66.9538,28.8255,98.5587,69.7333


------
## Cell 5 - Summary Statistics for Participant and Trial 
Group by participant and leg to get mean contact, peak flexion angle, and excursion. 


In [27]:
if results_dat.empty:
    print('No rseults to summarize - check cell 4 for errors.')

else:
    summary = (
        results_dat
        .groupby(['ID'])
        .agg(
            n_trials = ('Trial', 'count'),
            left_contact_mean = ('Left_Contact_Deg', 'mean'),
            left_peak_mean = ('Left_Peak_Deg', 'mean'),
            left_excursion_mean = ('Left_Excursion_Deg', 'mean'),
            right_contact_mean = ('Right_Contact_Deg', 'mean'),
            right_peak_mean = ('Right_Peak_Deg', 'mean'),
            right_excursion_mean = ('Right_Excursion_Deg', 'mean')
         )
        .round(2)
        .reset_index()
    )
display(summary)

,ID,n_trials,left_contact_mean,left_peak_mean,left_excursion_mean,right_contact_mean,right_peak_mean,right_excursion_mean
0,SHQ001,3,27.68,83.90,56.22,31.90,84.34,52.44
1,SHQ002,3,49.75,66.70,16.95,32.21,67.30,35.09
2,SHQ003,3,29.12,75.03,45.91,21.12,79.42,58.30
3,SHQ004,3,24.68,78.84,54.17,19.71,78.45,58.74
4,SHQ005,3,31.98,71.92,39.93,18.38,68.38,50.00
5,SHQ006,3,40.24,78.15,37.92,34.27,78.47,44.20
6,SHQ007,3,28.15,84.25,56.10,23.47,83.28,59.81
7,SHQ008,3,31.24,74.91,43.67,17.70,73.74,56.04
8,SHQ009,3,23.69,85.32,61.63,22.21,87.64,65.43
9,SHQ010,3,30.15,91.57,61.42,25.81,93.23,67.42


------
## Cell 6 - Save Results to an Excel file
A `Save As` dialog will open. The filename will be pre-filled with today's date. 

In [28]:
if results_dat.empty:
    print('No results to save - make sure you run Cell 4 first.')
else:
    today_str = datetime.now().strftime("%Y-%m-%d")
    default_name = f'SHQ_DLDropLanding_results_{today_str}.csv'
    save_path = pick_save_path(default_name=default_name)

    if not save_path:
        print('Save cancelled.')
    else:
        new_dat = results_dat
    
        if os.path.exists(save_path):
            existing_dat = pd.read_csv(save_path)
            combined_dat = pd.concat([existing_dat, new_dat], ignore_index=True)
            combined_dat = combined_dat.drop_duplicates(
                subset = ['ID', 'Trial'], keep = 'last'
            )
            print(f"Existing file found — appending. "
                  f"({len(existing_dat)} existing rows, "
                  f"{len(new_dat)} new rows, duplicates removed.)")
        else:
            combined_dat = new_dat
            print(f'Creating new file with {len(new_dat)} rows.')
        
        combined_dat.to_csv(save_path, index = False)
        print(f'Saved to: {save_path}')
        display(new_dat)

Creating new file with 62 rows.
Saved to: E:/Research/0-Mentoring/MoosbruggerJ/Data Collection/SHQ_DLDropLanding_results_2026-04-28.csv


,ID,Trial,Left_Contact_Deg,Left_Peak_Deg,Left_Excursion_Deg,Right_Contact_Deg,Right_Peak_Deg,Right_Excursion_Deg
0,SHQ001,Double_drop_1,40.2251,82.2780,42.0529,44.0786,77.9267,33.8481
1,SHQ001,Double_drop_2,19.7800,80.8605,61.0805,25.3088,83.4828,58.1740
2,SHQ001,Double_drop_3,23.0460,88.5583,65.5123,26.3041,91.5979,65.2939
3,SHQ002,Double_drop_1,46.0000,66.7963,20.7963,28.3870,66.4652,38.0782
4,SHQ002,Double_drop_2,53.1826,68.3347,15.1521,33.5196,69.5056,35.9860
...,...,...,...,...,...,...,...,...
57,SHQ021,Double_drop_2,35.0297,92.5102,57.4804,35.8187,92.8301,57.0114
58,SHQ021,Double_drop_3,41.0062,98.4308,57.4246,40.6825,101.0743,60.3918
59,SHQ022,Double_drop_1,29.8513,92.5240,62.6727,33.2738,93.5580,60.2842
60,SHQ022,Double_drop_2,31.3626,98.3164,66.9538,28.8255,98.5587,69.7333
